# Phase 1 — Understand the NASA IGBT dataset

## What we are investigating

Before analyzing degradation, we need to identify the experiments, device groups, file formats, recorded signals, and available time or failure information. Different folders in this archive contain different experiments; combining them without understanding their provenance could produce misleading conclusions.

**Why this matters in PHM:** degradation features and RUL labels are only meaningful when we know which device and experiment produced each observation, what the measurement represents, and whether an end-of-life event was actually recorded.

**This notebook is Phase 1 only.** It inventories files and inspects metadata and documentation. It does not clean signals, extract features, create RUL labels, or train a model. It never edits files in `data/raw/`.

### Initial findings from this extracted copy

- The archive is extracted under `data/raw/IGBTAgingData_04022009/`; no ZIP file was present under `data/raw/` when the project was inspected. The ZIP cell below checks again on your machine.
- The extracted copy contains 199 files: 12 MATLAB `.mat`, 135 `.csv`, 45 `.txt`, 5 legacy Word `.doc`, 1 PDF, and 1 Excel `.xls` file.
- The dataset README describes separate DC-gate aging, square-gate aging, square-gate aging with SMU characterization, and pristine-device characterization experiments. It says the square-gate-plus-SMU aging set contains four IGBTs (Devices 2–5); this is not a verified count of unique physical IGBTs across every experiment in the archive.
- The device aging log has sheets for Devices 2–5 and maps MAT filenames to recorded test events. Some log entries mention latch-up or inability to draw current; this is not the same as a complete, consistently timestamped EOL label for every run.
- The dataset README warns that some transient measurements are missing, collector-current measurements can drift by about 600 mA, and steady-state measurements are not properly scaled. Treat signal names and numerical values cautiously until their units and calibration are understood.

**Expected result:** a clear inventory, the actual nested variables in a representative MAT file, the spreadsheet's experiment notes, and a list of questions to resolve before data loading or signal analysis.

## 1. Locate the raw data and check for the original ZIP

`Path` represents filesystem paths in a platform-aware way. `rglob` searches below a directory. The ZIP is opened only to list its contents; this code does not extract or modify it.

In [ ]:
from collections import Counter
from pathlib import Path
from zipfile import ZipFile, is_zipfile

project_root = Path.cwd()
if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

raw_root = project_root / "data" / "raw"
if not raw_root.is_dir():
    raise FileNotFoundError(f"Could not find the raw-data folder: {raw_root}")

archives = sorted(raw_root.rglob("*.zip"))
print(f"Raw-data folder: {raw_root}")
if not archives:
    print("No ZIP archive found under data/raw/. The dataset appears to be extracted.")
else:
    for archive in archives:
        print(f"\nZIP archive: {archive.relative_to(project_root)}")
        if not is_zipfile(archive):
            print("  This file is not a valid ZIP archive.")
            continue
        with ZipFile(archive) as zip_file:
            members = [name for name in zip_file.namelist() if not name.endswith("/")]
            zip_extensions = Counter(Path(name).suffix.lower() or "[no extension]" for name in members)
            print(f"  Files: {len(members)}")
            print(f"  File types: {dict(sorted(zip_extensions.items()))}")
            print("  First 20 file paths:")
            for name in members[:20]:
                print("   ", name)

: 

## 2. Inventory the extracted files

An extension count tells us what readers may be needed; it does not tell us what each file means. The paths help connect files to their experiment folders.

In [ ]:
files = sorted(path for path in raw_root.rglob("*") if path.is_file())
extensions = Counter(path.suffix.lower() or "[no extension]" for path in files)

print(f"Total extracted files: {len(files)}")
print("File types:")
for extension, count in sorted(extensions.items()):
    print(f"  {extension}: {count}")

print("\nExperiment folders:")
data_root = raw_root / "IGBTAgingData_04022009" / "Data"
if not data_root.is_dir():
    candidates = [path for path in raw_root.rglob("Data") if path.is_dir()]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Could not identify one dataset Data folder under {raw_root}")
    data_root = candidates[0]
for folder in sorted(path for path in data_root.iterdir() if path.is_dir()):
    print(" ", folder.name)

### How many IGBTs are represented?

The dataset README identifies four devices (2–5) in the square-gate aging set that also includes SMU characterization. It separately describes a DC-gate aging experiment, another square-gate experiment, and pristine-device characterization. These experiment-group counts do **not** establish how many unique physical devices are represented across the entire archive. Use the experiment logs and documentation to keep groups separate until device identity is confirmed.

## 3. Inspect the aging MAT filenames without loading the signals

File names such as `Device2`, `Device2b`, and `Device2check` are labels in the archive. The aging log is needed to understand how those files relate to test events; do not infer continuation, failure, or calibration solely from the names.

In [ ]:
aging_data_folders = sorted(data_root.rglob("Aging Data"))
if not aging_data_folders:
    raise FileNotFoundError(f"No 'Aging Data' folder found under {data_root}")

for aging_folder in aging_data_folders:
    print(f"\nAging folder: {aging_folder.relative_to(raw_root)}")
    for path in sorted(aging_folder.rglob("*.mat")):
        print(f"  {path.relative_to(aging_folder)} ({path.stat().st_size:,} bytes)")

## 4. Inspect the structure of one actual MATLAB file

MAT files can contain nested structs and arrays rather than a simple rectangular table. `scipy.io.loadmat` reads a MATLAB file; the following cells first show the top-level variable names and only then inspect the `measurement` variable if it is present. The field names below come from the actual `Device2` file in this dataset.

In [ ]:
import numpy as np
from scipy.io import loadmat

mat_files = sorted(aging_data_folders[0].rglob("*.mat"))
representative_file = next(
    (path for path in mat_files if path.name.startswith("Device2 ")),
    mat_files[0],
)
mat_contents = loadmat(representative_file, squeeze_me=True, struct_as_record=False)

print(f"File: {representative_file.name}")
print("Top-level MATLAB variables:")
for name, value in mat_contents.items():
    if not name.startswith("__"):
        print(f"  {name}: Python type={type(value).__name__}, shape={getattr(value, 'shape', None)}")

if "measurement" not in mat_contents:
    raise KeyError("This selected MAT file does not contain a 'measurement' variable.")

measurement = mat_contents["measurement"]
print("\nFields inside measurement:", getattr(measurement, "_fieldnames", None))

In [ ]:
for group_name in getattr(measurement, "_fieldnames", []):
    group = getattr(measurement, group_name)
    records = np.atleast_1d(group)
    first_record = records[0] if len(records) else None
    print(f"\n{group_name}: {len(records)} record(s)")
    print("  First record fields:", getattr(first_record, "_fieldnames", None))

    if first_record is not None and hasattr(first_record, "timeDomain"):
        print("  timeDomain fields:", getattr(first_record.timeDomain, "_fieldnames", None))
    if first_record is not None and hasattr(first_record, "frequencyDomain"):
        print("  frequencyDomain fields:", getattr(first_record.frequencyDomain, "_fieldnames", None))

print("\nThese are nested measurement structures, not pandas columns. A MAT field name alone does not establish its unit, calibration, or degradation meaning.")

## 5. Check which time information is present

The aging MAT files contain dated records and `timeEpoch` values. The next cell summarizes the timestamps in steady-state records for each MAT segment. A timestamp is evidence of observation time; it is **not** automatically a cycle count, elapsed aging time, or failure time. The device aging log provides additional test-setting and event notes.

In [ ]:
for aging_folder in aging_data_folders:
    for path in sorted(aging_folder.rglob("*.mat")):
        contents = loadmat(path, squeeze_me=True, struct_as_record=False)
        if "measurement" not in contents or not hasattr(contents["measurement"], "steadyState"):
            print(f"{path.name}: no steadyState field to summarize")
            continue

        records = np.atleast_1d(contents["measurement"].steadyState)
        times = []
        for record in records:
            if hasattr(record, "timeEpoch"):
                value = float(np.asarray(record.timeEpoch).item())
                if np.isfinite(value) and value > 0:
                    times.append(value)

        if times:
            duration_minutes = (max(times) - min(times)) / 60
            print(f"{path.parent.name}/{path.name}: {len(records)} steadyState records; {len(times)} valid timestamps; span {duration_minutes:.1f} min")
        else:
            print(f"{path.parent.name}/{path.name}: {len(records)} steadyState records; no valid timeEpoch values found")

## 6. Read the device aging log

`Device Aging log.xls` is a legacy Excel `.xls` workbook. The `xlrd` dependency in `requirements.txt` lets pandas read this format. Each sheet records experiment information for a device. These notes are important context for matching file segments to test settings and events; they are not a machine-readable, uniform failure-label table.

In [ ]:
import pandas as pd

aging_log = next(data_root.rglob("Device Aging log.xls"), None)
if aging_log is None:
    print("Device Aging log.xls was not found. Check the extracted folder and rerun the inventory cells.")
else:
    workbook = pd.ExcelFile(aging_log, engine="xlrd")
    print("Aging-log worksheets:", workbook.sheet_names)
    for sheet_name in workbook.sheet_names:
        sheet = pd.read_excel(aging_log, sheet_name=sheet_name, header=None, engine="xlrd")
        print(f"\n--- {sheet_name}: {sheet.shape[0]} rows x {sheet.shape[1]} columns ---")
        display(sheet.dropna(axis=1, how="all"))

### What the documentation says about failure and candidate degradation signals

- The top-level README describes one DC-gate experiment as stopping after latch-up and thermal runaway, and points to changes in the collector-current tail during turn-off as a failure precursor for that experiment.
- The square-gate-plus-SMU README reports missing transient measurements, collector-current drift of about 600 mA, and incorrectly scaled steady-state data. Do not treat every current or temperature value as calibrated.
- The aging log records events for Devices 2–5. It explicitly notes latch-up for Device 3 and inability to draw current for Device 5. The notes for all files/devices are not a single standardized, independently verified EOL timestamp; an ordinary file end or a low-temperature `check` measurement is not by itself proof of failure.
- The archive includes electrical characterization described as threshold voltage, breakdown voltage, and leakage current. These could be degradation-sensitive measurements if they can be reliably aligned with each device's aging history; confirm that mapping before using them.
- The paper discusses a turn-off transient peak as a possible precursor in an early thermal-overstress experiment. That is a candidate to investigate, not proof that a particular variable is a validated health indicator in every folder.

**PHM distinction:** a candidate degradation indicator is a measurement worth investigating. It becomes a Health Indicator only after checking measurement quality, repeatability, and progression with aging. An EOL observation in a log is not automatically a reliable time label until its device, event, and timestamp are unambiguous.

## 7. Inspect one characterization CSV without inventing column names

The IGBT folders contain `Breakdown.csv`, `LeakageIV.csv`, and `Turn On.csv`. The observed CSV files have numeric data and no header row, so pandas' default behavior would incorrectly treat the first measurement as a header. `header=None` preserves it. The displayed column labels `0` and `1` are pandas positions, **not verified physical signal names or units**.

In [ ]:
igbt_characterization_folder = data_root / "SMU Data for new devices" / "IGBT-IRG4BC30K"
csv_files = sorted(igbt_characterization_folder.rglob("*.csv"))
print(f"IGBT characterization CSV files found: {len(csv_files)}")
for path in csv_files[:3]:
    print(f"\n{path.relative_to(data_root)}")
    sample = pd.read_csv(path, header=None, nrows=5)
    display(sample)

## Phase 1 summary and questions

Before proceeding to Phase 2, use the inventory and documents to answer:

1. Which experiment folder would you choose to study first, and why should it be kept separate from the other experiments?
2. Which MATLAB fields describe steady-state versus transient observations? Which measurement-quality limitations affect the square-gate-plus-SMU experiment?
3. Which devices have log notes that suggest a failure event? What additional evidence is needed before assigning an EOL timestamp to each device?
4. Which candidate degradation signals seem promising, and which measurement-quality or alignment questions would you investigate before trusting them?
5. Does the dataset provide a cycle count, and how is that different from the timestamps and test-setting notes we observed?

Send me your answers, the ZIP-search result, and any output that differs from the initial findings. We will resolve dataset ambiguity before moving to data loading. **No features or RUL labels are created in this notebook.**